# mboostrs — package tour

Componentwise functional gradient boosting in Rust, exposed to Python via PyO3.
Same algorithm, families, and `df → λ` rule as R's `mboost` 2.9.11 — no R runtime required.

Covered here:
1. glmboost — linear base-learners, risk path, coefficient trajectories.
2. cvrisk + mstop — pick the stopping iteration by cross-validation.
3. gamboost with bbs(...) P-splines — recover a nonlinear function.
4. Non-Gaussian families — Poisson, Binomial, Quantile.
5. predict on new data.

In [1]:
import numpy as np
import polars as pl
import pandas as pd
import altair as alt
import mboostrs as mb

print('mboost version:', mb.__version__)
rng = np.random.default_rng(0)

In [2]:
n = 500
p = 8
X = rng.normal(size=(n, p))
true_beta = np.array([1.5, -0.8, 0.0, 0.0, 0.6, 0.0, 0.0, -0.3])
y = X @ true_beta + rng.normal(scale=0.5, size=n)

data = pl.DataFrame({f'x{j+1}': X[:, j] for j in range(p)} | {'y': y})
formula = 'y ~ ' + ' + '.join(f'x{j+1}' for j in range(p))

## 1. glmboost — boosted linear model

At each of `mstop` iterations, the engine picks the single predictor whose least-squares fit to the current pseudo-residual reduces empirical risk most, then updates the additive predictor by `nu` times that contribution.

In [3]:
fit = mb.glmboost(formula, data=data, family=mb.Gaussian(),
                  control=mb.boost_control(mstop=300, nu=0.1))
print(fit)

In [4]:
selected_counts = pl.DataFrame({
    "learner": fit._learner_names,
    "selected": [fit.selected.count(n) for n in fit._learner_names],
})
display(selected_counts)

coef_dict = {k: v for k, v in fit.coefficients_.items() if k != '(Intercept)'}
truth_dict = {f'x{j+1}': true_beta[j] for j in range(p)}
coef_df = pl.DataFrame({
    "predictor": list(coef_dict.keys()),
    "boosted": list(coef_dict.values()),
    "truth": [truth_dict.get(k, float("nan")) for k in coef_dict],
})
display(coef_df)

In [5]:
risk_df = pd.DataFrame({
    "iteration": range(1, len(fit.risk_) + 1),
    "risk": fit.risk_
})
risk_chart = alt.Chart(risk_df).mark_line(strokeWidth=1.2).encode(
    x=alt.X("iteration:Q", title="iteration"),
    y=alt.Y("risk:Q", title="in-bag risk"),
).properties(title="Risk path (nu=0.1)", width=320, height=200)

# Coefficient trajectory: extract coef path from the model
n_coef = len(fit._learner_names)
traj_data = []
for k in range(1, len(fit.risk_) + 1):
    fit_k = fit[:k]
    for nm in fit._learner_names:
        val = fit_k.coefficients_.get(nm, 0.0)
        traj_data.append({"iteration": k, "coefficient": nm, "value": val})
traj_df = pd.DataFrame(traj_data)
traj_chart = alt.Chart(traj_df).mark_line(strokeWidth=1).encode(
    x=alt.X("iteration:Q", title="iteration"),
    y=alt.Y("value:Q", title="coefficient"),
    color=alt.Color("coefficient:N", legend=alt.Legend(columns=2)),
).properties(title="Coefficient paths", width=320, height=200)

alt.hconcat(risk_chart, traj_chart)

## 2. cvrisk + mstop — picking the stopping iteration

Training risk falls monotonically as iterations are added. Cross-validation estimates risk on held-out observations; select `mstop` by minimizing that out-of-bag risk path.

In [6]:
cvr = fit.cvrisk(type='kfold', B=10, seed=42)
print(cvr)

fit_opt = fit[: cvr.optimal_mstop]

# Compare in-bag vs CV risk
cv_df = pd.DataFrame({
    "iteration": range(1, len(cvr.aggregate) + 1),
    "CV risk": cvr.aggregate,
    "in-bag risk": fit.risk_[:len(cvr.aggregate)],
}).melt(id_vars=["iteration"], var_name="type", value_name="risk")

cv_chart = alt.Chart(cv_df).mark_line(strokeWidth=1.2).encode(
    x=alt.X("iteration:Q", title="iteration"),
    y=alt.Y("risk:Q", title="risk"),
    color=alt.Color("type:N", scale=alt.Scale(
        domain=["CV risk", "in-bag risk"],
        range=["#d62728", "#1f77b4"]
    )),
    strokeDash="type:N",
).properties(title=f"CV picks mstop = {cvr.optimal_mstop}", width=500, height=220)

vline = alt.Chart(pd.DataFrame({"x": [cvr.optimal_mstop]})).mark_rule(
    strokeDash=[4, 4], color="black"
).encode(x="x:Q")

cv_chart + vline

## 3. gamboost with P-spline base-learners (bbs)

For nonlinear effects, swap `bols(x)` for `bbs(x, df=...)`. The engine builds a B-spline basis with a difference penalty and converts `df` to ridge weight `λ` via the Demmler–Reinsch identity — the same solver R uses.

In [7]:
n = 600
x = rng.uniform(-3, 3, size=n)
z = rng.uniform(-3, 3, size=n)
f_true = np.sin(1.2 * x) + 0.5 * z
y = f_true + rng.normal(scale=0.25, size=n)

data = pl.DataFrame({'x': x, 'z': z, 'y': y})

fit = mb.gamboost(
    'y ~ bbs(x, df=4) + bols(z)',
    data=data,
    family=mb.Gaussian(),
    control=mb.boost_control(mstop=400, nu=0.1),
)
cvr = fit.cvrisk(type='kfold', B=5, seed=42)
fit_opt = fit[: max(cvr.optimal_mstop, 2)]
print(f'mstop* = {cvr.optimal_mstop}, final risk = {fit_opt.risk_[-1]:.4f}')

In [8]:
# Recover the nonlinear effect of x
grid = np.linspace(x.min(), x.max(), 200)
newdata = pl.DataFrame({'x': grid, 'z': np.zeros_like(grid)})
f_hat = fit_opt.predict(newdata=newdata, type='link') - fit_opt.offset_

true_curve = np.sin(1.2 * grid)
pdf1 = pd.DataFrame({"x": grid, "bbs(x) effect": f_hat, "truth: sin(1.2x)": true_curve})
pdf1 = pdf1.melt(id_vars=["x"], var_name="curve", value_name="y")
line = alt.Chart(pdf1).mark_line().encode(
    x=alt.X("x:Q", title="x"), y=alt.Y("y:Q", title="y"),
    color="curve:N", strokeDash="curve:N",
)
pts = alt.Chart(data.to_pandas()).mark_circle(
    size=8, opacity=0.3, color="gray"
).encode(x="x:Q", y="y:Q")
left = (pts + line).properties(title="Recovered nonlinear effect", width=300, height=200)

# Risk path with CV
risk_df = pd.DataFrame({"iteration": range(1, len(fit.risk_) + 1), "training risk": fit.risk_})
cv_df = pd.DataFrame({"iteration": range(1, len(cvr.aggregate) + 1), "CV risk": cvr.aggregate})
risk_chart = alt.Chart(risk_df).mark_line(strokeWidth=1).encode(
    x="iteration:Q", y=alt.Y("training risk:Q", title="risk")
).properties(title="Risk paths", width=300, height=200)
cv_line = alt.Chart(cv_df).mark_line(strokeWidth=1.4, color="red").encode(
    x="iteration:Q", y=alt.Y("CV risk:Q", title="")
)
vline = alt.Chart(pd.DataFrame({"x": [cvr.optimal_mstop]})).mark_rule(
    strokeDash=[4, 4], color="black"
).encode(x="x:Q")

alt.hconcat(left, risk_chart + cv_line + vline)

## 4. Other families

Available: `Gaussian`, `Binomial`, `Poisson`, `GammaReg`, `Laplace`, `Quantile(tau)`, `Expectile(tau)`, `Huber(delta)`. Each supplies the same `(offset, negative_gradient, risk)` triple as R.

In [9]:
n = 400
x1 = rng.normal(size=n)
x2 = rng.normal(size=n)
eta = 0.7 + 0.6*x1 - 0.4*x2
y = rng.poisson(np.exp(eta))
data = pl.DataFrame({'y': y.astype(float), 'x1': x1, 'x2': x2})

fit_pois = mb.glmboost('y ~ x1 + x2', data=data, family=mb.Poisson(),
                       control=mb.boost_control(mstop=200, nu=0.1))
offset = np.log(data['y'].mean())

# Risk path
risk_p = pd.DataFrame({"iteration": range(1, len(fit_pois.risk_)+1), "Poisson risk": fit_pois.risk_})
chart_p = alt.Chart(risk_p).mark_line().encode(
    x="iteration:Q", y=alt.Y("Poisson risk:Q", title="risk")
).properties(title="Poisson risk path", width=500, height=180)

# Coefficient recovery
truth_map = {'(Intercept)': 0.7, 'x1': 0.6, 'x2': -0.4}
coef_df = pl.DataFrame({
    "term": list(fit_pois.coefficients_.keys()),
    "boosted": list(fit_pois.coefficients_.values()),
    "truth": [truth_map.get(k, float("nan")) for k in fit_pois.coefficients_],
})
icpt = fit_pois.coefficients_['(Intercept)']
print(f'offset (log mean y) = {offset:.3f}')
print(f'offset + intercept = {offset:.3f} + ({icpt:+.3f}) = {offset+icpt:.3f} \u2192 true intercept = 0.700')
display(coef_df)
chart_p

In [10]:
n = 600
x = rng.uniform(-2, 2, size=n)
y = 1.5*x + (0.5 + 0.4*np.abs(x))*rng.normal(size=n)
data = pl.DataFrame({'y': y, 'x': x})

taus = [0.1, 0.5, 0.9]
preds = {}
risks = {}
grid = np.linspace(x.min(), x.max(), 200)
for tau in taus:
    fit_q = mb.gamboost(
        'y ~ bbs(x, df=4)',
        data=data,
        family=mb.Quantile(tau=tau),
        control=mb.boost_control(mstop=400, nu=0.1),
    )
    preds[tau] = fit_q.predict(newdata=pl.DataFrame({'x': grid}), type='link')
    risks[tau] = fit_q.risk_

# Quantile curves
rows = []
for tau in taus:
    for xi, pred in zip(grid, preds[tau]):
        rows.append({"x": xi, "y": pred, "\u03c4": f"\u03c4 = {tau}"})
qdf = pd.DataFrame(rows)
pts = alt.Chart(data.to_pandas()).mark_circle(size=8, opacity=0.25, color="gray").encode(x="x:Q", y="y:Q")
lines = alt.Chart(qdf).mark_line(strokeWidth=1.8).encode(
    x="x:Q", y="y:Q", color="\u03c4:N"
)
left = (pts + lines).properties(title="Quantile curves \u03c4 \u2208 {0.1, 0.5, 0.9}", width=350, height=240)

# Risk paths
risk_rows = []
for tau in taus:
    for i, r in enumerate(risks[tau]):
        risk_rows.append({"iteration": i+1, "risk": r, "\u03c4": f"\u03c4 = {tau}"})
rdf = pd.DataFrame(risk_rows)
right = alt.Chart(rdf).mark_line(strokeWidth=1).encode(
    x="iteration:Q", y=alt.Y("risk:Q", title="empirical risk"),
    color="\u03c4:N", strokeDash="\u03c4:N",
).properties(title="Risk paths by quantile", width=350, height=240)

alt.hconcat(left, right)

## 5. Prediction on new data

`predict(newdata=...)` accepts a polars DataFrame, a pandas DataFrame, or a dict of arrays. `type='link'` returns the additive predictor; `type='response'` applies the inverse link.

In [11]:
n = 800
x = rng.normal(size=n)
z = rng.normal(size=n)
eta = 0.5*x - 0.3*z
p_true = 1.0 / (1.0 + np.exp(-2*eta))
y_bin = (rng.uniform(size=n) < p_true).astype(float)
data = pl.DataFrame({'y': y_bin, 'x': x, 'z': z})

fit_bin = mb.glmboost('y ~ x + z', data=data, family=mb.Binomial(),
                      control=mb.boost_control(mstop=200, nu=0.1))

x_grid = np.linspace(-3, 3, 200)
newdata = pl.DataFrame({'x': x_grid, 'z': np.zeros_like(x_grid)})
p_hat = fit_bin.predict(newdata=newdata, type='response')

p_true_grid = 1.0 / (1.0 + np.exp(-2 * 0.5 * x_grid))
bdf = pd.DataFrame({"x": x_grid, "truth (z=0)": p_true_grid, "boosted probability": p_hat})
base = alt.Chart(bdf).transform_fold(
    ["truth (z=0)", "boosted probability"], as_=["curve", "probability"]
).mark_line().encode(
    x=alt.X("x:Q", title="x"),
    y=alt.Y("probability:Q", title="P(y=1 | x, z=0)"),
    color="curve:N",
    strokeDash="curve:N",
)
pts = alt.Chart(data.to_pandas()).mark_circle(
    size=6, opacity=0.15
).encode(x="x:Q", y="y:Q")
(pts + base).properties(
    title="Binomial glmboost — predict(type=\"response\")",
    width=500, height=220
)

---

See `02_r_parity.ipynb` to verify numerical parity with R, and `03_performance.ipynb` for timing comparisons.